# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ali0369/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

##Finding 1 — The Freshness Multiplier

The paper reports that content older than 365 days and refreshed within 30 days had a higher average health score and more impressions than comparable older content that had not been recently refreshed.

My methodology question is about how the refreshed and untouched groups were selected. Pages chosen for refresh may already have stronger historical demand, more strategic importance, or better baseline performance. Because this is an observational comparison, the difference cannot by itself prove that refreshing caused all of the observed improvement.

A stronger validation design could compare refreshed and unrefreshed pages with similar age, baseline impressions, topic, and previous performance. A before-and-after analysis with a matched comparison group would provide stronger evidence.

The paper already presents this finding carefully as a measured portfolio pattern. My question is not whether the finding is useful, but whether the comparison design fully separates the effect of refreshing from differences that existed before the refresh.

##Finding 2 — Feature importance for predicting health score

The paper's **Random Forest** analysis found that average position and impressions were among the strongest predictors of health score.

My methodology question is about target overlap. The paper defines health score partly using impressions, position, CTR, and scroll depth. If the model uses some of these same variables as input features, then high feature importance may partly reflect that the model is predicting a composite score using variables already used to construct that score.

This means the feature importance result should be interpreted as descriptive rather than causal. It does not show that changing average position or impressions independently causes a change in health score.

A stronger validation design could predict an external outcome that is not directly constructed from the input features, or remove overlapping variables and compare how model performance changes.

The paper appropriately notes this limitation and describes the importance analysis as model behavior rather than a standalone optimization rule.

In [1]:
print("Two research findings and methodology questions documented.")

Two research findings and methodology questions documented.


## 2. My model under an honest split (before/after)

My Week-5 model used a time-based split: observations before March 22 were used for training and observations from March 22 onward were used for testing.

That split respects time, but the same client can appear in both training and test data. Because pages from the same client may share patterns, I want to check whether the model still performs similarly when entire clients are kept together.

For this audit, I will compare the original time-based result with a stricter client-grouped split. The grouped split keeps each `client_hash_id` in only one side of the evaluation.

I will use the same Logistic Regression model, the same four Week-5 features, and the same target. This makes the comparison about validation design rather than changing the model.

The results will be treated as measured model performance on these splits, not as proof that the model will perform the same way on future clients.

In [2]:
# Loading Data
from datasets import load_dataset
ds = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", streaming=True, split="train")



README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

In [3]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


In [4]:
# Connecting DuckDB
import duckdb
con = duckdb.connect()
con.execute(f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
""")
REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
print("DuckDB connection ready.")

DuckDB connection ready.


In [5]:
ml_data = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,

    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN SUM(gsc_clicks) * 100.0 / SUM(gsc_impressions)
        ELSE 0
    END AS ctr,

    AVG(
        CASE
            WHEN gsc_avg_position > 0
            THEN gsc_avg_position
            ELSE NULL
        END
    ) AS avg_position,

    MAX(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END)
        AS gsc_available

FROM {REL}

WHERE gsc_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id
""").df()

print("ML rows:", len(ml_data))
print("Columns:", ml_data.columns.tolist())

ml_data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

ML rows: 176738
Columns: ['client_hash_id', 'content_hash_id', 'impressions', 'clicks', 'ctr', 'avg_position', 'gsc_available']


,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position,gsc_available
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,77.0,0.0,0.000000,4.888929,1
1,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,10849.0,22.0,0.202784,8.240351,1
2,client_62f4a7e64f5e0096,content_e689bc511192751a,61.0,0.0,0.000000,7.061594,1
3,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,705.0,1.0,0.141844,6.155424,1
4,client_62f4a7e64f5e0096,content_40b10da45f4c1cb5,50.0,0.0,0.000000,14.343567,1


In [6]:
feature_cols = [
    "impressions",
    "clicks",
    "ctr",
    "avg_position"
]

X = ml_data[feature_cols].copy()

print("Features:")
print(feature_cols)

X.head()

Features:
['impressions', 'clicks', 'ctr', 'avg_position']


,impressions,clicks,ctr,avg_position
0,77.0,0.0,0.000000,4.888929
1,10849.0,22.0,0.202784,8.240351
2,61.0,0.0,0.000000,7.061594
3,705.0,1.0,0.141844,6.155424
4,50.0,0.0,0.000000,14.343567


In [7]:
# April data
APR_REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
)
"""

print("April source configured.")

April source configured.


In [8]:
# March daily features
march_daily = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,

    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN SUM(gsc_clicks) * 100.0 / SUM(gsc_impressions)
        ELSE 0
    END AS ctr,

    AVG(
        CASE
            WHEN gsc_avg_position > 0
            THEN gsc_avg_position
            ELSE NULL
        END
    ) AS avg_position

FROM {REL}

WHERE gsc_data_available IS TRUE

GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
""").df()

print("March daily rows:", len(march_daily))
march_daily.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March daily rows: 3611061


,report_date,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position
0,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125.0,1.0,0.80000,4.928000
1,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11.0,0.0,0.00000,2.272727
2,2026-03-01,client_73cda7b4e4f265ea,content_36c36abc7650d7af,239.0,1.0,0.41841,7.347280
3,2026-03-01,client_73cda7b4e4f265ea,content_a7da352b73b02668,191.0,0.0,0.00000,7.832461
4,2026-03-01,client_73cda7b4e4f265ea,content_1855a661b4d36130,14.0,0.0,0.00000,3.428571


In [9]:
# April 1–7 outcome data
april_daily = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,

    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN SUM(gsc_clicks) * 100.0 / SUM(gsc_impressions)
        ELSE 0
    END AS ctr,

    AVG(
        CASE
            WHEN gsc_avg_position > 0
            THEN gsc_avg_position
            ELSE NULL
        END
    ) AS avg_position

FROM {APR_REL}

WHERE
    gsc_data_available IS TRUE
    AND report_date <= DATE '2026-04-07'

GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
""").df()

print("April outcome rows:", len(april_daily))
april_daily.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

April outcome rows: 880921


,report_date,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position
0,2026-04-01,client_62f4a7e64f5e0096,content_76c1f31e2b38f054,13.0,0.0,0.0,20.000000
1,2026-04-01,client_62f4a7e64f5e0096,content_ffc5ab4b34aab1f8,16.0,0.0,0.0,3.437500
2,2026-04-01,client_62f4a7e64f5e0096,content_9739856fc83dc1ca,63.0,0.0,0.0,3.809524
3,2026-04-01,client_62f4a7e64f5e0096,content_3d1dc691a3502105,315.0,0.0,0.0,2.244444
4,2026-04-01,client_62f4a7e64f5e0096,content_9d28af4f99c5e67b,224.0,0.0,0.0,5.370536


In [10]:
#Combine March & April
import pandas as pd
all_daily = pd.concat(
    [march_daily, april_daily],
    ignore_index=True
)

all_daily = all_daily.sort_values(
    ["client_hash_id", "content_hash_id", "report_date"]
).reset_index(drop=True)

print("Combined rows:", len(all_daily))
print("First date:", all_daily["report_date"].min())
print("Last date:", all_daily["report_date"].max())

Combined rows: 4491982
First date: 2026-03-01 00:00:00
Last date: 2026-04-07 00:00:00


In [11]:
# Create the future 7-day label
con.register("daily_data", all_daily)

future_labels = con.sql("""
    SELECT
        a.report_date,
        a.client_hash_id,
        a.content_hash_id,
        CASE
            WHEN MAX(
                CASE
                    WHEN b.avg_position <= 10
                         AND b.ctr < 2
                    THEN 1
                    ELSE 0
                END
            ) = 1
            THEN 1
            ELSE 0
        END AS future_low_ctr_strong_position
    FROM daily_data a
    LEFT JOIN daily_data b
        ON a.client_hash_id = b.client_hash_id
        AND a.content_hash_id = b.content_hash_id
        AND b.report_date > a.report_date
        AND b.report_date <= a.report_date + INTERVAL 7 DAY
    WHERE a.report_date >= DATE '2026-03-01'
      AND a.report_date < DATE '2026-04-01'
    GROUP BY
        a.report_date,
        a.client_hash_id,
        a.content_hash_id
""").df()

future_labels.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,future_low_ctr_strong_position
0,2026-03-26,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,0
1,2026-03-31,client_0797ff3a1fc9a6a5,content_04c67f3541177192,0
2,2026-03-29,client_0797ff3a1fc9a6a5,content_12890868e4cdac06,1
3,2026-03-31,client_0797ff3a1fc9a6a5,content_167472cd0802a8f3,1
4,2026-03-31,client_0797ff3a1fc9a6a5,content_20346a450ede60c6,1


In [12]:
all_daily = all_daily.drop(
    columns=["future_low_ctr_strong_position"],
    errors="ignore"
)

all_daily = all_daily.merge(
    future_labels,
    on=["report_date", "client_hash_id", "content_hash_id"],
    how="left"
)

all_daily["future_low_ctr_strong_position"] = (
    all_daily["future_low_ctr_strong_position"]
    .fillna(0)
    .astype(int)
)

print(
    all_daily["future_low_ctr_strong_position"]
    .value_counts()
)

future_low_ctr_strong_position
1    2724270
0    1767712
Name: count, dtype: int64


In [13]:
model_data = all_daily[
    (all_daily["report_date"] >= "2026-03-01") &
    (all_daily["report_date"] <= "2026-03-31")
].copy()

model_data = model_data[
    model_data["report_date"] <= "2026-03-31"
].copy()

print("Model rows:", len(model_data))
print(
    model_data["future_low_ctr_strong_position"]
    .value_counts()
)

Model rows: 3611061
future_low_ctr_strong_position
1    2724270
0     886791
Name: count, dtype: int64


In [17]:

import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

feature_cols = [
    "impressions",
    "clicks",
    "ctr",
    "avg_position"
]

target_col = "future_low_ctr_strong_position"

print("Features:", feature_cols)
print("Target:", target_col)

# Checking missing values before training
print("\nMissing values:")
print(model_data[feature_cols].isna().sum())

Features: ['impressions', 'clicks', 'ctr', 'avg_position']
Target: future_low_ctr_strong_position

Missing values:
impressions          0
clicks               0
ctr                  0
avg_position    163189
dtype: int64


In [15]:

train_mask = model_data["report_date"] < "2026-03-22"
test_mask = model_data["report_date"] >= "2026-03-22"

X = model_data[feature_cols].copy()
y = model_data[target_col].astype(int)

X_train = X.loc[train_mask]
X_test = X.loc[test_mask]

y_train = y.loc[train_mask]
y_test = y.loc[test_mask]

print("Original time-based split")
print("-------------------------")
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training clients:", model_data.loc[train_mask, "client_hash_id"].nunique())
print("Test clients:", model_data.loc[test_mask, "client_hash_id"].nunique())

overlap_clients = set(
    model_data.loc[train_mask, "client_hash_id"]
).intersection(
    set(model_data.loc[test_mask, "client_hash_id"])
)

print("Clients appearing in both train and test:", len(overlap_clients))

Original time-based split
-------------------------
Training rows: 2366418
Test rows: 1244643
Training clients: 45
Test clients: 46
Clients appearing in both train and test: 44


In [18]:
time_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

time_model.fit(X_train, y_train)

time_probability = time_model.predict_proba(X_test)[:, 1]
time_prediction = time_model.predict(X_test)

print("Original time-based model trained.")

Original time-based model trained.


In [19]:
def evaluate_predictions(y_true, prediction, probability, name):
    return {
        "split": name,
        "accuracy": accuracy_score(y_true, prediction),
        "precision": precision_score(y_true, prediction, zero_division=0),
        "recall": recall_score(y_true, prediction, zero_division=0),
        "f1": f1_score(y_true, prediction, zero_division=0),
        "roc_auc": roc_auc_score(y_true, probability)
    }


before_result = evaluate_predictions(
    y_test,
    time_prediction,
    time_probability,
    "Week-5 time split"
)

before_result

{'split': 'Week-5 time split',
 'accuracy': 0.827574653936912,
 'precision': 0.9029060034683587,
 'recall': 0.8641560360348836,
 'f1': 0.8831061450323053,
 'roc_auc': np.float64(0.8530528723543639)}

In [20]:
print("Confusion matrix — original time split")
print(confusion_matrix(y_test, time_prediction))

Confusion matrix — original time split
[[219378  87174]
 [127434 810657]]


In [21]:
unique_clients = model_data["client_hash_id"].dropna().unique()

rng = np.random.RandomState(42)
rng.shuffle(unique_clients)

split_point = int(len(unique_clients) * 0.70)

train_clients = set(unique_clients[:split_point])
test_clients = set(unique_clients[split_point:])

group_train_mask = model_data["client_hash_id"].isin(train_clients)
group_test_mask = model_data["client_hash_id"].isin(test_clients)

X_group_train = model_data.loc[group_train_mask, feature_cols].copy()
X_group_test = model_data.loc[group_test_mask, feature_cols].copy()

y_group_train = model_data.loc[group_train_mask, target_col].astype(int)
y_group_test = model_data.loc[group_test_mask, target_col].astype(int)

print("Client-grouped split")
print("-------------------")
print("Training rows:", len(X_group_train))
print("Test rows:", len(X_group_test))
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))

group_overlap = train_clients.intersection(test_clients)

print("Clients appearing in both groups:", len(group_overlap))

Client-grouped split
-------------------
Training rows: 3205845
Test rows: 405216
Training clients: 32
Test clients: 15
Clients appearing in both groups: 0


In [22]:
print("Training target distribution:")
print(y_group_train.value_counts())

print("\nTest target distribution:")
print(y_group_test.value_counts())

Training target distribution:
future_low_ctr_strong_position
1    2406326
0     799519
Name: count, dtype: int64

Test target distribution:
future_low_ctr_strong_position
1    317944
0     87272
Name: count, dtype: int64


In [24]:

group_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

group_model.fit(X_group_train, y_group_train)

group_probability = group_model.predict_proba(X_group_test)[:, 1]
group_prediction = group_model.predict(X_group_test)

print("Client-grouped model trained.")

Client-grouped model trained.


In [25]:
group_result = evaluate_predictions(
    y_group_test,
    group_prediction,
    group_probability,
    "Client-grouped split"
)

group_result

{'split': 'Client-grouped split',
 'accuracy': 0.8450998973387033,
 'precision': 0.9003338521010091,
 'recall': 0.9024859723724933,
 'f1': 0.9014086276875134,
 'roc_auc': np.float64(0.8478596313903455)}

In [26]:
comparison = pd.DataFrame([
    before_result,
    group_result
])

comparison

,split,accuracy,precision,recall,f1,roc_auc
0,Week-5 time split,0.827575,0.902906,0.864156,0.883106,0.853053
1,Client-grouped split,0.845100,0.900334,0.902486,0.901409,0.847860


In [27]:
print("Confusion matrix — client-grouped split")
print(confusion_matrix(y_group_test, group_prediction))

Confusion matrix — client-grouped split
[[ 55508  31764]
 [ 31004 286940]]


## 3. Leakage audit

I checked the final four Week-5 features for information that could make the model see the target directly or indirectly.

The target is `future_low_ctr_strong_position`, which is based on performance observed after each March observation.

The four model features are impressions, clicks, CTR, and average position. These features describe the current observation and are not calculated from the future seven-day target.

I will also check whether any feature contains missing or unexpected values and whether the target is directly present in the feature columns.

In [28]:
print("Feature columns:")
print(feature_cols)

print("\nTarget column:")
print(target_col)

print("\nTarget included as a feature:", target_col in feature_cols)

Feature columns:
['impressions', 'clicks', 'ctr', 'avg_position']

Target column:
future_low_ctr_strong_position

Target included as a feature: False


In [29]:
# Check for missing, infinite, and unusual values.

print("Missing values:")
print(model_data[feature_cols].isna().sum())

print("\nInfinite values:")
print(np.isinf(model_data[feature_cols].select_dtypes(include=[np.number])).sum())

Missing values:
impressions          0
clicks               0
ctr                  0
avg_position    163189
dtype: int64

Infinite values:
impressions     0
clicks          0
ctr             0
avg_position    0
dtype: int64


In [30]:
print("Model date range:")
print(model_data["report_date"].min(), "to", model_data["report_date"].max())

print("\nTarget value counts:")
print(model_data[target_col].value_counts())

print("\nFeature summary:")
print(model_data[feature_cols].describe())

Model date range:
2026-03-01 00:00:00 to 2026-03-31 00:00:00

Target value counts:
future_low_ctr_strong_position
1    2724270
0     886791
Name: count, dtype: int64

Feature summary:
        impressions        clicks           ctr  avg_position
count  3.611061e+06  3.611061e+06  3.611061e+06  3.447872e+06
mean   7.772164e+01  2.275874e-01  3.080748e-01  1.657573e+01
std    2.498747e+02  1.277267e+00  3.009151e+00  2.001265e+01
min    1.000000e+00  0.000000e+00  0.000000e+00  3.106340e-04
25%    4.000000e+00  0.000000e+00  0.000000e+00  4.153846e+00
50%    1.600000e+01  0.000000e+00  0.000000e+00  8.000000e+00
75%    6.200000e+01  0.000000e+00  0.000000e+00  2.135490e+01
max    4.008400e+04  2.740000e+02  1.000000e+02  4.980000e+02


In [31]:


leakage_checks = {
    "target_in_features": target_col in feature_cols,
    "future_target_column_used_as_feature": target_col in feature_cols,
    "missing_feature_values": int(model_data[feature_cols].isna().sum().sum()),
    "infinite_feature_values": int(
        np.isinf(
            model_data[feature_cols].select_dtypes(include=[np.number])
        ).sum().sum()
    )
}

leakage_checks

{'target_in_features': False,
 'future_target_column_used_as_feature': False,
 'missing_feature_values': 163189,
 'infinite_feature_values': 0}

In [34]:
print("Final leakage audit")
print("-------------------")

print("Target included in features:", target_col in feature_cols)
print("Missing feature values:", leakage_checks["missing_feature_values"])
print("Infinite feature values:", leakage_checks["infinite_feature_values"])

print(
    "\nConclusion: The target column is not included in the model features. "
    "The four features describe the current observation, while the target "
    "is calculated from later observations."
)

Final leakage audit
-------------------
Target included in features: False
Missing feature values: 163189
Infinite feature values: 0

Conclusion: The target column is not included in the model features. The four features describe the current observation, while the target is calculated from later observations.


## 4. Claim rewrite

### Original claim

My strongest claim was that average position and impressions are strong predictors of content health.

### Safer claim

In this dataset, average position and impressions were measured as important inputs in the model's prediction of the health-related target. This is an observed model pattern and should be treated as directional decision-support, not evidence that changing these variables will directly cause better performance.

In [32]:

print("Validation comparison")
print("=====================")

print("\nOriginal Week-5 time split:")
for key, value in before_result.items():
    if key != "split":
        print(f"{key}: {value:.4f}")

print("\nClient-grouped split:")
for key, value in group_result.items():
    if key != "split":
        print(f"{key}: {value:.4f}")


Validation comparison

Original Week-5 time split:
accuracy: 0.8276
precision: 0.9029
recall: 0.8642
f1: 0.8831
roc_auc: 0.8531

Client-grouped split:
accuracy: 0.8451
precision: 0.9003
recall: 0.9025
f1: 0.9014
roc_auc: 0.8479


In [33]:


metric_cols = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc"
]

validation_difference = {}

for metric in metric_cols:
    validation_difference[metric] = (
        group_result[metric] - before_result[metric]
    )

validation_difference

{'accuracy': 0.01752524340179129,
 'precision': -0.002572151367349562,
 'recall': 0.03832993633760973,
 'f1': 0.018302482655208063,
 'roc_auc': np.float64(-0.0051932409640184085)}

### Final interpretation

The stricter client-grouped validation provides a useful check against relying only on the original time-based result.

Any difference between the two splits is treated as a measured validation difference. It does not establish causation and does not guarantee future performance.

The model is therefore best described as decision-support for identifying patterns in this dataset rather than as a causal or guaranteed prediction system.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.